In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import rasterio
import numpy as np
import os, glob

base = '/content/drive/MyDrive/FloodProject/data/sen1floods11'

# Pick one chip from each folder to inspect
samples = {
    'S1Weak'          : glob.glob(f'{base}/train/S1Weak/*.tif')[0],
    'S2Weak'          : glob.glob(f'{base}/train/S2Weak/*.tif')[0],
    'Label'           : glob.glob(f'{base}/train/S2IndexLabelWeak/*.tif')[0],
    'S1Hand'          : glob.glob(f'{base}/val/S1Hand/*.tif')[0],
    'S2Hand'          : glob.glob(f'{base}/val/S2Hand/*.tif')[0],
    'LabelHand'       : glob.glob(f'{base}/val/LabelHand/*.tif')[0],
    'DEM'             : glob.glob(f'{base}/dem/chips/*.tif')[0],
    'Temporal_India'  : f'{base}/temporal/India_TEMPORAL.tif',
}

print(f"{'Source':20s}  {'Bands':>5}  {'Shape (HxW)':>14}  {'dtype':>8}  {'CRS':>12}  {'Res (m approx)':>16}  File")
print('-' * 115)
for name, path in samples.items():
    with rasterio.open(path) as src:
        h, w   = src.height, src.width
        res_x  = abs(src.transform.a)
        res_y  = abs(src.transform.e)
        print(f"{name:20s}  {src.count:>5}  {h:>6} x {w:<6}  {str(src.dtypes[0]):>8}  {str(src.crs.to_epsg()):>12}  {res_x:.6f} x {res_y:.6f}  {os.path.basename(path)}")

Source                Bands     Shape (HxW)     dtype           CRS    Res (m approx)  File
-------------------------------------------------------------------------------------------------------------------
S1Weak                    2     512 x 512      float32          4326  0.000090 x 0.000090  Mekong_8226942_S1Weak.tif
S2Weak                   13     512 x 512       uint16          4326  0.000090 x 0.000090  Mekong_8320459_S2Weak.tif
Label                     1     512 x 512        int16          4326  0.000090 x 0.000090  Mekong_8369112_S2IndexLabelWeak.tif
S1Hand                    2     512 x 512      float32          4326  0.000090 x 0.000090  India_1018317_S1Hand.tif
S2Hand                   13     512 x 512        int16          4326  0.000090 x 0.000090  India_1017769_S2Hand.tif
LabelHand                 1     512 x 512        int16          4326  0.000090 x 0.000090  India_1018317_LabelHand.tif
DEM                       4     512 x 512      float32          4326  0.000090 x

In [ ]:
# 1. Check which S2 bands are in S2Weak/S2Hand
# (sen1floods11 includes all 13 bands but we only want B2,B3,B4,B8,B11,B12)
import rasterio, glob

path = glob.glob('/content/drive/MyDrive/FloodProject/data/sen1floods11/train/S2Weak/*.tif')[0]
with rasterio.open(path) as src:
    print("S2Weak band descriptions:", src.descriptions)
    print("S2Weak dtype:", src.dtypes[0])
    print("S2Weak nodata:", src.nodata)

# 2. Check label values
import numpy as np
label_path = glob.glob('/content/drive/MyDrive/FloodProject/data/sen1floods11/train/S2IndexLabelWeak/*.tif')[0]
with rasterio.open(label_path) as src:
    arr = src.read(1)
    print("\nLabel unique values:", np.unique(arr))
    print("Label nodata:", src.nodata)

# 3. Check DEM band descriptions
dem_path = glob.glob('/content/drive/MyDrive/FloodProject/data/sen1floods11/dem/chips/*.tif')[0]
with rasterio.open(dem_path) as src:
    print("\nDEM band descriptions:", src.descriptions)
    print("DEM nodata:", src.nodata)

S2Weak band descriptions: ('B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B10', 'B11', 'B12')
S2Weak dtype: uint16
S2Weak nodata: None

Label unique values: [-1  0  1]
Label nodata: None

DEM band descriptions: (None, None, None, None)
DEM nodata: None


***Pre-processing***

In [ ]:
# rasterio is not pre-installed in Colab
import subprocess
subprocess.run(['pip', 'install', 'rasterio', '--quiet'], check=True)
print('rasterio ready')

rasterio ready


***Configuration***

In [ ]:
import os, json, glob, random
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
from tqdm import tqdm

# ── Paths ─────────────────────────────────────────────────────────────────
BASE        = '/content/drive/MyDrive/FloodProject/data/sen1floods11'
OUT_BASE    = '/content/drive/MyDrive/FloodProject/data/preprocessed'

TRAIN_S1    = f'{BASE}/train/S1Weak'
TRAIN_S2    = f'{BASE}/train/S2Weak'
TRAIN_LABEL = f'{BASE}/train/S2IndexLabelWeak'
VAL_S1      = f'{BASE}/val/S1Hand'
VAL_S2      = f'{BASE}/val/S2Hand'
VAL_LABEL   = f'{BASE}/val/LabelHand'
DEM_DIR     = f'{BASE}/dem/chips'
TEMPORAL_DIR= f'{BASE}/temporal'
STATS_PATH  = f'{OUT_BASE}/norm_stats.json'

# ── Band indices (1-based, as rasterio uses) ──────────────────────────────
S1_BANDS  = [1, 2]           # VV, VH
S2_BANDS  = [2, 3, 4, 8, 11, 12]  # B2,B3,B4,B8,B11,B12
DEM_BANDS = [1, 2, 3, 4]     # Elevation, Slope, TWI, HAND
# NDWI and NDVI are computed, not read from file

# Indices into the S2_BANDS list for index computation
# S2_BANDS = [B2, B3, B4, B8, B11, B12] → indices 0..5
IDX_B3  = 1   # Green  (for NDWI)
IDX_B4  = 2   # Red    (for NDVI)
IDX_B8  = 3   # NIR    (for NDWI + NDVI)

# ── Temporal interleaving: band index in GeoTIFF for each day ─────────────
# Band order: chirps_day00(1), era5_day00(2), chirps_day01(3), era5_day01(4) ...
N_DAYS = 15

# ── Sampling ──────────────────────────────────────────────────────────────
CAMBODIA_SAMPLE_N = 400
RANDOM_SEED       = 42

# ── Countries and their flood dates (for temporal lookup) ─────────────────
# Must match the prefix in chip filenames e.g. 'India_1018317_S1Hand.tif'
COUNTRY_FLOOD_DATE = {
    'India'     : '2016-08-12',
    'Pakistan'  : '2017-06-28',
    'Sri-Lanka' : '2017-05-30',
    'Cambodia'  : '2018-08-05',
    'Bolivia'   : '2018-02-15',
    'Colombia'  : '2018-08-22',
}

# Create output directories
for split in ['train', 'val']:
    os.makedirs(f'{OUT_BASE}/{split}', exist_ok=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print('Configuration loaded.')
print(f'Output directory: {OUT_BASE}')

Configuration loaded.
Output directory: /content/drive/MyDrive/FloodProject/data/preprocessed


***Helper Functions***

In [ ]:
def get_chip_id(filepath):
    """Extract chip ID from filename.
    e.g. 'India_1018317_S1Hand.tif' -> 'India_1018317'
    """
    return '_'.join(os.path.basename(filepath).split('_')[:2])


def get_country(chip_id):
    """Extract country prefix from chip_id.
    e.g. 'India_1018317' -> 'India'
    """
    return chip_id.split('_')[0]


def read_bands(path, band_indices):
    """Read specific bands from a GeoTIFF. Returns (C, H, W) float32 array.
    band_indices: list of 1-based band numbers.
    """
    with rasterio.open(path) as src:
        arr = src.read(band_indices).astype(np.float32)  # (C, H, W)
    return arr


def read_label(path):
    """Read label chip. Returns (H, W) int16 array.
    Values: 0=no-flood, 1=flood, -1=invalid.
    """
    with rasterio.open(path) as src:
        return src.read(1).astype(np.int16)  # (H, W)


def compute_index(a, b):
    """Compute normalised difference index (a-b)/(a+b).
    Returns float32 array, clipped to [-1, 1]. Division by zero -> 0.
    """
    denom = a + b
    with np.errstate(invalid='ignore', divide='ignore'):
        idx = np.where(denom == 0, 0.0, (a - b) / denom)
    return np.clip(idx, -1.0, 1.0).astype(np.float32)


def reproject_temporal_to_chip(temporal_path, chip_s1_path):
    """
    Reproject the country-level temporal GeoTIFF to exactly match
    the chip's pixel grid (same CRS, transform, shape as the S1 chip).

    Returns array of shape (15, 2, H, W) float32:
      dim0 = timestep (0=flood_date-14, 14=flood_date)
      dim1 = variable (0=CHIRPS, 1=ERA5)
      dim2, dim3 = spatial (H, W) = 512, 512

    Resampling: bilinear (appropriate for continuous contextual variables).
    """
    with rasterio.open(chip_s1_path) as chip_src:
        dst_crs       = chip_src.crs
        dst_transform = chip_src.transform
        dst_height    = chip_src.height
        dst_width     = chip_src.width

    with rasterio.open(temporal_path) as temp_src:
        n_bands = temp_src.count  # 30
        src_crs       = temp_src.crs
        src_transform = temp_src.transform

        # Read all 30 bands at once
        src_data = temp_src.read().astype(np.float32)  # (30, H_src, W_src)
        src_nodata = temp_src.nodata

    # Reproject each of the 30 bands to chip grid
    reprojected = np.zeros((n_bands, dst_height, dst_width), dtype=np.float32)
    for b in range(n_bands):
        reproject(
            source      = src_data[b],
            destination = reprojected[b],
            src_transform  = src_transform,
            src_crs        = src_crs,
            dst_transform  = dst_transform,
            dst_crs        = dst_crs,
            resampling     = Resampling.bilinear,
            src_nodata     = src_nodata,
            dst_nodata     = 0.0,
        )

    # Reshape from (30, H, W) interleaved → (15, 2, H, W)
    # Band order in file: chirps_day00(0), era5_day00(1), chirps_day01(2), era5_day01(3)...
    temporal_tensor = np.zeros((N_DAYS, 2, dst_height, dst_width), dtype=np.float32)
    for t in range(N_DAYS):
        temporal_tensor[t, 0] = reprojected[t * 2]      # CHIRPS
        temporal_tensor[t, 1] = reprojected[t * 2 + 1]  # ERA5

    return temporal_tensor  # (15, 2, 512, 512)


def fill_nan_inf(arr, fill_value=0.0):
    """Replace NaN and Inf with fill_value in-place."""
    arr[~np.isfinite(arr)] = fill_value
    return arr


print('Helper functions defined.')

Helper functions defined.


***Build Chip Manifest***

Builds a list of all chip IDs per split, then applies Cambodia downsampling to the train split

In [ ]:
def build_manifest(s1_dir):
    """Return sorted list of chip_ids found in s1_dir."""
    paths = sorted(glob.glob(f'{s1_dir}/*.tif'))
    return [get_chip_id(p) for p in paths]


# Full manifests
train_chips_all = build_manifest(TRAIN_S1)
val_chips       = build_manifest(VAL_S1)

# Separate Cambodia train chips for downsampling
cambodia_chips  = [c for c in train_chips_all if get_country(c) == 'Cambodia']
other_chips     = [c for c in train_chips_all if get_country(c) != 'Cambodia']

# Stratified sample: shuffle with fixed seed then take first N
random.shuffle(cambodia_chips)
cambodia_sampled = cambodia_chips[:CAMBODIA_SAMPLE_N]

train_chips = other_chips + cambodia_sampled

# ── Summary ───────────────────────────────────────────────────────────────
from collections import Counter
train_countries = Counter(get_country(c) for c in train_chips)
val_countries   = Counter(get_country(c) for c in val_chips)

print(f'TRAIN chips: {len(train_chips)}')
for country, n in sorted(train_countries.items()):
    print(f'  {country:12s}: {n}')

print(f'\nVAL chips: {len(val_chips)}')
for country, n in sorted(val_countries.items()):
    print(f'  {country:12s}: {n}')

print(f'\nCambodia: {len(cambodia_chips)} total → {len(cambodia_sampled)} sampled')

TRAIN chips: 3017
  Bolivia     : 224
  Colombia    : 534
  India       : 467
  Mekong      : 1353
  Pakistan    : 249
  Sri-Lanka   : 190

VAL chips: 168
  India       : 68
  Mekong      : 30
  Pakistan    : 28
  Sri-Lanka   : 42

Cambodia: 0 total → 0 sampled


***Preprocess One Chip (Dry Run)***

Test the full pipeline on a single chip before running on all chips

In [ ]:
def process_chip(chip_id, s1_dir, s2_dir, label_dir):
    """
    Process one chip: read all sources, align, stack, compute indices.
    Returns:
        spatial  : (14, 512, 512) float32  — NOT yet normalized
        temporal : (15,  2, 512, 512) float32  — NOT yet normalized
        label    : (512, 512) int16
    """
    country = get_country(chip_id)

    # ── File paths ────────────────────────────────────────────────────────
    # Filenames follow pattern: {chip_id}_{suffix}.tif
    # We glob to find the exact filename since suffix varies (S1Weak vs S1Hand etc.)
    s1_path    = glob.glob(f'{s1_dir}/{chip_id}_*.tif')[0]
    s2_path    = glob.glob(f'{s2_dir}/{chip_id}_*.tif')[0]
    label_path = glob.glob(f'{label_dir}/{chip_id}_*.tif')[0]
    dem_path   = glob.glob(f'{DEM_DIR}/{chip_id}_*.tif')[0]
    temp_path  = f'{TEMPORAL_DIR}/{country}_TEMPORAL.tif'

    # ── Read S1 (2 bands: VV, VH) ─────────────────────────────────────────
    s1 = read_bands(s1_path, S1_BANDS)          # (2, 512, 512) float32
    fill_nan_inf(s1)

    # ── Read S2 (6 bands: B2,B3,B4,B8,B11,B12) ───────────────────────────
    s2 = read_bands(s2_path, S2_BANDS)          # (6, 512, 512) float32 (cast from uint16)
    fill_nan_inf(s2)

    # ── Compute NDWI = (B3 - B8) / (B3 + B8) ────────────────────────────
    ndwi = compute_index(s2[IDX_B3], s2[IDX_B8])  # (512, 512)

    # ── Compute NDVI = (B8 - B4) / (B8 + B4) ────────────────────────────
    ndvi = compute_index(s2[IDX_B8], s2[IDX_B4])  # (512, 512)

    # ── Read DEM (4 bands: Elevation, Slope, TWI, HAND) ──────────────────
    dem = read_bands(dem_path, DEM_BANDS)        # (4, 512, 512) float32
    fill_nan_inf(dem)

    # ── Reproject Temporal to chip grid ──────────────────────────────────
    temporal = reproject_temporal_to_chip(temp_path, s1_path)  # (15, 2, 512, 512)
    fill_nan_inf(temporal)

    # ── Stack spatial tensor (14 channels) ───────────────────────────────
    # Order: S1(0-1), S2(2-7), DEM(8-11), NDWI(12), NDVI(13)
    spatial = np.concatenate([
        s1,                          # (2, 512, 512)
        s2,                          # (6, 512, 512)
        dem,                         # (4, 512, 512)
        ndwi[np.newaxis],            # (1, 512, 512)
        ndvi[np.newaxis],            # (1, 512, 512)
    ], axis=0)                       # → (14, 512, 512)

    assert spatial.shape  == (14, 512, 512), f'Spatial shape mismatch: {spatial.shape}'
    assert temporal.shape == (15, 2, 512, 512), f'Temporal shape mismatch: {temporal.shape}'
    assert spatial.dtype  == np.float32
    assert temporal.dtype == np.float32

    # ── Read label ────────────────────────────────────────────────────────
    label = read_label(label_path)               # (512, 512) int16

    return spatial, temporal, label


# ── Dry run on first train chip ───────────────────────────────────────────
test_chip = train_chips[0]
print(f'Dry run on: {test_chip}')
sp, tp, lb = process_chip(test_chip, TRAIN_S1, TRAIN_S2, TRAIN_LABEL)

print(f'  spatial  shape : {sp.shape}  dtype={sp.dtype}')
print(f'  temporal shape : {tp.shape}  dtype={tp.dtype}')
print(f'  label    shape : {lb.shape}  dtype={lb.dtype}')
print(f'  label values   : {np.unique(lb)}')
print(f'  spatial  NaN?  : {np.isnan(sp).any()}')
print(f'  temporal NaN?  : {np.isnan(tp).any()}')
print()
print('Per-channel spatial stats (pre-norm):')
ch_names = ['VV','VH','B2','B3','B4','B8','B11','B12','Elev','Slope','TWI','HAND','NDWI','NDVI']
print(f'  {"Ch":>4}  {"Name":>6}  {"Min":>10}  {"Max":>10}  {"Mean":>10}  {"Std":>10}')
for i, name in enumerate(ch_names):
    ch = sp[i]
    print(f'  {i:>4}  {name:>6}  {ch.min():>10.3f}  {ch.max():>10.3f}  {ch.mean():>10.3f}  {ch.std():>10.3f}')

Dry run on: Bolivia_1009032
  spatial  shape : (14, 512, 512)  dtype=float32
  temporal shape : (15, 2, 512, 512)  dtype=float32
  label    shape : (512, 512)  dtype=int16
  label values   : [-1  0  1]
  spatial  NaN?  : False
  temporal NaN?  : False

Per-channel spatial stats (pre-norm):
    Ch    Name         Min         Max        Mean         Std
     0      VV     -48.271      -0.521     -11.597       5.534
     1      VH     -50.016      -6.700     -17.630       5.776
     2      B2     739.000    5088.000    1156.281     560.393
     3      B3     486.000    5092.000    1120.354     579.903
     4      B4     262.000    5316.000     824.228     637.251
     5      B8     187.000    5928.000    2281.915    1296.464
     6     B11       4.000      15.000       8.355       1.631
     7     B12      22.000    5426.000    1311.886     906.350
     8    Elev     138.000     149.991     139.844       1.004
     9   Slope       0.006       0.006       0.006       0.000
    10     TWI  

***Compute Normalization Statistics***

Statistics are computed on the **training split only**, then applied to both train and val.

This prevents data leakage from the val split into the normalization.

Uses Welford's online algorithm to compute mean and std without loading all chips into memory simultaneously.

In [ ]:
# Welford online mean/variance — memory efficient, no need to store all chips
# Computes per-channel statistics over all pixels across all training chips

N_SPATIAL  = 14
N_TEMPORAL = 2   # CHIRPS and ERA5 (each normalized independently across all timesteps)

# Spatial accumulators
sp_count = np.zeros(N_SPATIAL, dtype=np.float64)
sp_mean  = np.zeros(N_SPATIAL, dtype=np.float64)
sp_M2    = np.zeros(N_SPATIAL, dtype=np.float64)

# Temporal accumulators (one per variable, across all timesteps and pixels)
tp_count = np.zeros(N_TEMPORAL, dtype=np.float64)
tp_mean  = np.zeros(N_TEMPORAL, dtype=np.float64)
tp_M2    = np.zeros(N_TEMPORAL, dtype=np.float64)


def welford_update_channel(count, mean, M2, values):
    """Update Welford accumulators with a flat array of new values."""
    for x in values.ravel():
        count  += 1
        delta   = x - mean
        mean   += delta / count
        delta2  = x - mean
        M2     += delta * delta2
    return count, mean, M2


def welford_update_batch(counts, means, M2s, arr):
    """
    Vectorized Welford update for a (C, ...) array.
    Updates accumulators for each channel c independently.
    """
    C = arr.shape[0]
    flat = arr.reshape(C, -1).astype(np.float64)  # (C, N_pixels)
    n    = flat.shape[1]
    batch_mean = flat.mean(axis=1)                 # (C,)
    batch_var  = flat.var(axis=1)                  # (C,)
    # Parallel Welford merge
    total = counts + n
    delta = batch_mean - means
    new_mean = (counts * means + n * batch_mean) / total
    new_M2   = M2s + batch_var * n + delta**2 * counts * n / total
    return total, new_mean, new_M2


print(f'Computing normalization statistics over {len(train_chips)} training chips...')
errors = []

for chip_id in tqdm(train_chips):
    try:
        sp, tp, lb = process_chip(chip_id, TRAIN_S1, TRAIN_S2, TRAIN_LABEL)

        # Only use valid pixels for spatial stats (exclude invalid label pixels)
        valid_mask = (lb != -1)  # (512, 512) bool
        sp_valid   = sp[:, valid_mask]   # (14, N_valid)

        sp_count, sp_mean, sp_M2 = welford_update_batch(
            sp_count, sp_mean, sp_M2, sp_valid
        )

        # Temporal: shape (15, 2, 512, 512) → treat all timesteps together per variable
        # tp[:, 0] = CHIRPS across all 15 days, tp[:, 1] = ERA5 across all 15 days
        for v in range(N_TEMPORAL):
            vals = tp[:, v, :, :].ravel().astype(np.float64)  # (15*512*512,)
            n    = len(vals)
            bm   = vals.mean()
            bv   = vals.var()
            total = tp_count[v] + n
            delta = bm - tp_mean[v]
            tp_mean[v] = (tp_count[v] * tp_mean[v] + n * bm) / total
            tp_M2[v]   = tp_M2[v] + bv * n + delta**2 * tp_count[v] * n / total
            tp_count[v] = total

    except Exception as e:
        errors.append((chip_id, str(e)))

# Compute final std from M2
sp_std = np.sqrt(sp_M2 / sp_count)
tp_std = np.sqrt(tp_M2 / tp_count)

# Replace zero std with 1 to avoid division by zero (constant channels)
sp_std = np.where(sp_std == 0, 1.0, sp_std)
tp_std = np.where(tp_std == 0, 1.0, tp_std)

print(f'\nDone. Errors: {len(errors)}')
if errors:
    for chip_id, err in errors[:5]:
        print(f'  {chip_id}: {err}')

print('\nSpatial normalization stats (per channel):')
ch_names = ['VV','VH','B2','B3','B4','B8','B11','B12','Elev','Slope','TWI','HAND','NDWI','NDVI']
print(f'  {"Ch":>3}  {"Name":>6}  {"Mean":>10}  {"Std":>10}')
for i, name in enumerate(ch_names):
    print(f'  {i:>3}  {name:>6}  {sp_mean[i]:>10.4f}  {sp_std[i]:>10.4f}')

tp_names = ['CHIRPS', 'ERA5']
print('\nTemporal normalization stats (per variable):')
for i, name in enumerate(tp_names):
    print(f'  {name:>8}  mean={tp_mean[i]:.4f}  std={tp_std[i]:.4f}')

Computing normalization statistics over 3017 training chips...


100%|██████████| 3017/3017 [3:25:25<00:00,  4.09s/it]


Done. Errors: 1353
  Mekong_1000963: /content/drive/MyDrive/FloodProject/data/sen1floods11/temporal/Mekong_TEMPORAL.tif: No such file or directory
  Mekong_1023244: /content/drive/MyDrive/FloodProject/data/sen1floods11/temporal/Mekong_TEMPORAL.tif: No such file or directory
  Mekong_1026202: /content/drive/MyDrive/FloodProject/data/sen1floods11/temporal/Mekong_TEMPORAL.tif: No such file or directory
  Mekong_1027619: /content/drive/MyDrive/FloodProject/data/sen1floods11/temporal/Mekong_TEMPORAL.tif: No such file or directory
  Mekong_102939: /content/drive/MyDrive/FloodProject/data/sen1floods11/temporal/Mekong_TEMPORAL.tif: No such file or directory

Spatial normalization stats (per channel):
   Ch    Name        Mean         Std
    0      VV    -10.0610      4.6996
    1      VH    -17.0087      5.4536
    2      B2   1338.4577    424.6429
    3      B3   1249.1518    434.4354
    4      B4   1037.9719    551.8071
    5      B8   2331.0000    892.5690
    6     B11     74.4431    12

***Save Normalization Statistics***

In [ ]:
stats = {
    'spatial': {
        'channels': ['VV','VH','B2','B3','B4','B8','B11','B12','Elev','Slope','TWI','HAND','NDWI','NDVI'],
        'mean': sp_mean.tolist(),
        'std' : sp_std.tolist(),
    },
    'temporal': {
        'variables': ['CHIRPS', 'ERA5'],
        'mean': tp_mean.tolist(),
        'std' : tp_std.tolist(),
    },
    'metadata': {
        'n_train_chips'       : len(train_chips),
        'cambodia_sample_n'   : CAMBODIA_SAMPLE_N,
        'random_seed'         : RANDOM_SEED,
        'computed_on'         : 'train split only',
        'valid_pixels_only'   : True,
    }
}

with open(STATS_PATH, 'w') as f:
    json.dump(stats, f, indent=2)

print(f'Saved normalization stats to: {STATS_PATH}')

Saved normalization stats to: /content/drive/MyDrive/FloodProject/data/preprocessed/norm_stats.json


In [ ]:
import json

with open('/content/drive/MyDrive/FloodProject/data/preprocessed/norm_stats.json') as f:
    stats = json.load(f)

print('Spatial stats:')
for i, (name, mean, std) in enumerate(zip(
    stats['spatial']['channels'],
    stats['spatial']['mean'],
    stats['spatial']['std']
)):
    print(f'  {i:>2}  {name:>6}  mean={mean:>10.4f}  std={std:>10.4f}')

print('\nTemporal stats:')
for name, mean, std in zip(
    stats['temporal']['variables'],
    stats['temporal']['mean'],
    stats['temporal']['std']
):
    print(f'  {name:>8}  mean={mean:.4f}  std={std:.4f}')

print('\nMetadata:', stats['metadata'])

Spatial stats:
   0      VV  mean=  -10.0610  std=    4.6996
   1      VH  mean=  -17.0087  std=    5.4536
   2      B2  mean= 1338.4577  std=  424.6429
   3      B3  mean= 1249.1518  std=  434.4354
   4      B4  mean= 1037.9719  std=  551.8071
   5      B8  mean= 2331.0000  std=  892.5690
   6     B11  mean=   74.4431  std=  129.7152
   7     B12  mean= 1615.3577  std=  763.0594
   8    Elev  mean=  118.8102  std=  102.1684
   9   Slope  mean=    0.1547  std=    0.1837
  10     TWI  mean=    9.8925  std=    3.1343
  11    HAND  mean=   17.4972  std=   43.6591
  12    NDWI  mean=   -0.2655  std=    0.2333
  13    NDVI  mean=    0.3621  std=    0.2717

Temporal stats:
    CHIRPS  mean=9.6488  std=13.0074
      ERA5  mean=0.3897  std=0.1411

Metadata: {'n_train_chips': 3017, 'cambodia_sample_n': 400, 'random_seed': 42, 'computed_on': 'train split only', 'valid_pixels_only': True}


In [ ]:
import shutil
total, used, free = shutil.disk_usage('/content')
print(f'Local /content — total: {total/1e9:.1f} GB  used: {used/1e9:.1f} GB  free: {free/1e9:.1f} GB')

Local /content — total: 115.7 GB  used: 40.8 GB  free: 74.8 GB


In [ ]:
import time, rasterio, glob

files = glob.glob('/content/drive/MyDrive/FloodProject/data/sen1floods11/train/S1Weak/*.tif')[:10]

t0 = time.time()
for f in files:
    with rasterio.open(f) as src:
        src.read()
elapsed = time.time() - t0

print(f'Read 10 S1 chips in {elapsed:.1f}s → {elapsed/10:.2f}s per chip')
print(f'Estimated time for 1306 chips (stats pass): {1306 * elapsed/10 / 60:.0f} mins')
print(f'Estimated time for 1444 chips (save pass) : {1444 * elapsed/10 / 60:.0f} mins')

Read 10 S1 chips in 73.5s → 7.35s per chip
Estimated time for 1306 chips (stats pass): 160 mins
Estimated time for 1444 chips (save pass) : 177 mins


In [ ]:
import subprocess, os, glob

# Check what's missing
src_files = glob.glob('/content/drive/MyDrive/FloodProject/data/sen1floods11/**/*.tif', recursive=True)
dst_files = glob.glob('/content/sen1floods11/**/*.tif', recursive=True)

print(f'Source : {len(src_files)} tif files')
print(f'Dest   : {len(dst_files)} tif files')
print(f'Missing: {len(src_files) - len(dst_files)} files')

Source : 12746 tif files
Dest   : 12746 tif files
Missing: 0 files


In [ ]:
import os, glob

base_src = '/content/drive/MyDrive/FloodProject/data/sen1floods11'
base_dst = '/content/sen1floods11'

checks = [
    'train/S1Weak',
    'train/S2Weak',
    'train/S2IndexLabelWeak',
    'val/S1Hand',
    'val/S2Hand',
    'val/LabelHand',
    'dem/chips',
    'temporal',
]

print(f'{"Folder":35s}  {"Drive":>6}  {"Local":>6}  {"Match"}')
print('-' * 60)
all_ok = True
for subdir in checks:
    src_n = len(glob.glob(f'{base_src}/{subdir}/*.tif'))
    dst_n = len(glob.glob(f'{base_dst}/{subdir}/*.tif'))
    match = '✅' if src_n == dst_n else '❌'
    if src_n != dst_n:
        all_ok = False
    print(f'{subdir:35s}  {src_n:>6}  {dst_n:>6}  {match}')

print()
print('All folders match ✅' if all_ok else 'MISMATCH DETECTED ❌ — do not proceed')

Folder                                Drive   Local  Match
------------------------------------------------------------
train/S1Weak                           3017    3017  ✅
train/S2Weak                           3017    3017  ✅
train/S2IndexLabelWeak                 3017    3017  ✅
val/S1Hand                              168     168  ✅
val/S2Hand                              168     168  ✅
val/LabelHand                           168     168  ✅
dem/chips                              3185    3185  ✅
temporal                                  6       6  ✅

All folders match ✅


***Copying Input Data to Local SSD***

Copying from Drive to `/content/` local SSD before processing.

Reads will be ~10x faster than Drive, reducing Cell 8 from ~12 hrs to ~30 mins.

**Run once per session.** If the session restarts, re-run this cell before Cell 8.

In [ ]:
import subprocess, shutil, time

SRC  = '/content/drive/MyDrive/FloodProject/data/sen1floods11'
DEST = '/content/sen1floods11'

if os.path.exists(DEST):
    print(f'Already exists: {DEST} — skipping copy.')
else:
    print('Copying sen1floods11 to local SSD (~36 GB, 5–10 mins)...')
    t0 = time.time()
    subprocess.run(['cp', '-r', SRC, DEST], check=True)
    elapsed = time.time() - t0
    print(f'Done in {elapsed/60:.1f} mins.')

# Verify key directories exist
for subdir in ['train/S1Weak', 'train/S2Weak', 'train/S2IndexLabelWeak',
               'val/S1Hand', 'val/S2Hand', 'val/LabelHand',
               'dem/chips', 'temporal']:
    path = f'{DEST}/{subdir}'
    n = len(glob.glob(f'{path}/*.tif'))
    print(f'  {subdir:35s}: {n} tif files')

# Update BASE to point to local SSD
BASE_LOCAL      = DEST
TRAIN_S1_L      = f'{BASE_LOCAL}/train/S1Weak'
TRAIN_S2_L      = f'{BASE_LOCAL}/train/S2Weak'
TRAIN_LABEL_L   = f'{BASE_LOCAL}/train/S2IndexLabelWeak'
VAL_S1_L        = f'{BASE_LOCAL}/val/S1Hand'
VAL_S2_L        = f'{BASE_LOCAL}/val/S2Hand'
VAL_LABEL_L     = f'{BASE_LOCAL}/val/LabelHand'
DEM_DIR_L       = f'{BASE_LOCAL}/dem/chips'
TEMPORAL_DIR_L  = f'{BASE_LOCAL}/temporal'
print('\nLocal SSD paths set.')

Already exists: /content/sen1floods11 — skipping copy.
  train/S1Weak                       : 3017 tif files
  train/S2Weak                       : 3017 tif files
  train/S2IndexLabelWeak             : 3017 tif files
  val/S1Hand                         : 168 tif files
  val/S2Hand                         : 168 tif files
  val/LabelHand                      : 168 tif files
  dem/chips                          : 3185 tif files
  temporal                           : 6 tif files

Local SSD paths set.


***Fix Mekong→Cambodia Mapping & Recompute Stats***

Cell 5 had 1353 errors because chip filenames use `Mekong_` prefix but
the temporal file is `Cambodia_TEMPORAL.tif`. This cell fixes that mapping,
rebuilds the manifest, and recomputes stats correctly from local SSD.

**Replaces the stats computed in Cell 5** — overwrites `norm_stats.json`.

In [ ]:
# ── Country name map: chip filename prefix → temporal filename prefix ──
# Chip files: Mekong_XXXXXXX_S1Weak.tif
# Temporal  : Cambodia_TEMPORAL.tif
COUNTRY_NAME_MAP = {'Mekong': 'Cambodia'}

def get_temporal_country(chip_id):
    """Map chip country prefix to temporal file country name."""
    country = get_country(chip_id)
    return COUNTRY_NAME_MAP.get(country, country)


def process_chip_local(chip_id, s1_dir, s2_dir, label_dir):
    """
    Same as process_chip() but uses local SSD paths and the
    corrected Mekong→Cambodia temporal mapping.
    """
    temporal_country = get_temporal_country(chip_id)

    s1_path    = glob.glob(f'{s1_dir}/{chip_id}_*.tif')[0]
    s2_path    = glob.glob(f'{s2_dir}/{chip_id}_*.tif')[0]
    label_path = glob.glob(f'{label_dir}/{chip_id}_*.tif')[0]
    dem_path   = glob.glob(f'{DEM_DIR_L}/{chip_id}_*.tif')[0]
    temp_path  = f'{TEMPORAL_DIR_L}/{temporal_country}_TEMPORAL.tif'

    s1 = read_bands(s1_path, S1_BANDS)
    fill_nan_inf(s1)

    s2 = read_bands(s2_path, S2_BANDS)
    fill_nan_inf(s2)

    ndwi = compute_index(s2[IDX_B3], s2[IDX_B8])
    ndvi = compute_index(s2[IDX_B8], s2[IDX_B4])

    dem = read_bands(dem_path, DEM_BANDS)
    fill_nan_inf(dem)

    temporal = reproject_temporal_to_chip(temp_path, s1_path)
    fill_nan_inf(temporal)

    spatial = np.concatenate([
        s1,
        s2,
        dem,
        ndwi[np.newaxis],
        ndvi[np.newaxis],
    ], axis=0)  # (14, 512, 512)

    assert spatial.shape  == (14, 512, 512)
    assert temporal.shape == (15, 2, 512, 512)

    label = read_label(label_path)
    return spatial, temporal, label


# ── Rebuild manifest from local SSD ──────────────────────────────────────
train_chips_all_l = build_manifest(TRAIN_S1_L)
val_chips_l       = build_manifest(VAL_S1_L)

cambodia_chips_l  = [c for c in train_chips_all_l if get_country(c) == 'Mekong']
other_chips_l     = [c for c in train_chips_all_l if get_country(c) != 'Mekong']

random.seed(RANDOM_SEED)
random.shuffle(cambodia_chips_l)
cambodia_sampled_l = cambodia_chips_l[:CAMBODIA_SAMPLE_N]
train_chips_l      = other_chips_l + cambodia_sampled_l

from collections import Counter
print(f'TRAIN: {len(train_chips_l)} chips')
for c, n in sorted(Counter(get_country(x) for x in train_chips_l).items()):
    print(f'  {c:12s}: {n}')
print(f'VAL  : {len(val_chips_l)} chips')

# ── Recompute stats from local SSD ───────────────────────────────────────
sp_count = np.zeros(N_SPATIAL,  dtype=np.float64)
sp_mean  = np.zeros(N_SPATIAL,  dtype=np.float64)
sp_M2    = np.zeros(N_SPATIAL,  dtype=np.float64)
tp_count = np.zeros(N_TEMPORAL, dtype=np.float64)
tp_mean  = np.zeros(N_TEMPORAL, dtype=np.float64)
tp_M2    = np.zeros(N_TEMPORAL, dtype=np.float64)

print(f'\nRecomputing stats over {len(train_chips_l)} chips from local SSD...')
errors = []

for chip_id in tqdm(train_chips_l):
    try:
        sp, tp, lb = process_chip_local(chip_id, TRAIN_S1_L, TRAIN_S2_L, TRAIN_LABEL_L)
        valid_mask = (lb != -1)
        sp_valid   = sp[:, valid_mask]
        sp_count, sp_mean, sp_M2 = welford_update_batch(sp_count, sp_mean, sp_M2, sp_valid)
        for v in range(N_TEMPORAL):
            vals  = tp[:, v, :, :].ravel().astype(np.float64)
            n     = len(vals)
            bm    = vals.mean()
            bv    = vals.var()
            total = tp_count[v] + n
            delta = bm - tp_mean[v]
            tp_mean[v] = (tp_count[v] * tp_mean[v] + n * bm) / total
            tp_M2[v]   = tp_M2[v] + bv * n + delta**2 * tp_count[v] * n / total
            tp_count[v] = total
    except Exception as e:
        errors.append((chip_id, str(e)))

sp_std = np.sqrt(sp_M2 / sp_count)
tp_std = np.sqrt(tp_M2 / tp_count)
sp_std = np.where(sp_std == 0, 1.0, sp_std)
tp_std = np.where(tp_std == 0, 1.0, tp_std)

print(f'Done. Errors: {len(errors)}')
if errors:
    for chip_id, err in errors[:5]:
        print(f'  {chip_id}: {err}')

print('\nSpatial stats (per channel):')
ch_names = ['VV','VH','B2','B3','B4','B8','B11','B12','Elev','Slope','TWI','HAND','NDWI','NDVI']
print(f'  {"Ch":>3}  {"Name":>6}  {"Mean":>10}  {"Std":>10}')
for i, name in enumerate(ch_names):
    print(f'  {i:>3}  {name:>6}  {sp_mean[i]:>10.4f}  {sp_std[i]:>10.4f}')
print('\nTemporal stats:')
for i, name in enumerate(['CHIRPS','ERA5']):
    print(f'  {name:>8}  mean={tp_mean[i]:.4f}  std={tp_std[i]:.4f}')

# ── Overwrite norm_stats.json with corrected stats ────────────────────────
stats = {
    'spatial': {
        'channels': ch_names,
        'mean': sp_mean.tolist(),
        'std' : sp_std.tolist(),
    },
    'temporal': {
        'variables': ['CHIRPS', 'ERA5'],
        'mean': tp_mean.tolist(),
        'std' : tp_std.tolist(),
    },
    'metadata': {
        'n_train_chips'     : len(train_chips_l),
        'cambodia_sample_n' : CAMBODIA_SAMPLE_N,
        'random_seed'       : RANDOM_SEED,
        'computed_on'       : 'train split only',
        'valid_pixels_only' : True,
        'mekong_fix'        : 'Mekong chips mapped to Cambodia_TEMPORAL.tif',
    }
}
with open(STATS_PATH, 'w') as f:
    json.dump(stats, f, indent=2)
print(f'\nOverwrote {STATS_PATH} with corrected stats.')

TRAIN: 2064 chips
  Bolivia     : 224
  Colombia    : 534
  India       : 467
  Mekong      : 400
  Pakistan    : 249
  Sri-Lanka   : 190
VAL  : 168 chips

Recomputing stats over 2064 chips from local SSD...


100%|██████████| 2064/2064 [27:54<00:00,  1.23it/s]


Done. Errors: 0

Spatial stats (per channel):
   Ch    Name        Mean         Std
    0      VV    -10.3565      4.9099
    1      VH    -17.5690      6.1993
    2      B2   1308.5457    404.9387
    3      B3   1241.3179    420.8296
    4      B4   1051.9785    550.8950
    5      B8   2279.6153    914.6513
    6     B11     60.8675    118.8358
    7     B12   1605.4039    827.9655
    8    Elev     98.6885    100.1739
    9   Slope      0.1298      0.1718
   10     TWI      9.7797      2.8388
   11    HAND     15.3691     39.5855
   12    NDWI     -0.2522      0.2518
   13    NDVI      0.3409      0.2927

Temporal stats:
    CHIRPS  mean=9.3180  std=12.1788
      ERA5  mean=0.3946  std=0.1291

Overwrote /content/drive/MyDrive/FloodProject/data/preprocessed/norm_stats.json with corrected stats.


***Process & Save All Chips***

Reads from local SSD, normalizes, saves `.npy` files to Drive.
Skips chips already saved — **safe to re-run after disconnection**.

**Estimated time: ~30–45 mins** (reads from SSD, writes to Drive).

In [ ]:
# Load corrected stats
with open(STATS_PATH) as f:
    stats = json.load(f)

sp_mean_norm = np.array(stats['spatial']['mean'], dtype=np.float32)
sp_std_norm  = np.array(stats['spatial']['std'],  dtype=np.float32)
tp_mean_norm = np.array(stats['temporal']['mean'], dtype=np.float32)
tp_std_norm  = np.array(stats['temporal']['std'],  dtype=np.float32)


def normalize_spatial(sp):
    return (sp - sp_mean_norm[:, None, None]) / sp_std_norm[:, None, None]


def normalize_temporal(tp):
    return (tp - tp_mean_norm[None, :, None, None]) / tp_std_norm[None, :, None, None]


def save_chip(chip_id, spatial, temporal, label, out_dir):
    np.save(f'{out_dir}/{chip_id}_spatial.npy',  spatial)
    np.save(f'{out_dir}/{chip_id}_temporal.npy', temporal)
    np.save(f'{out_dir}/{chip_id}_label.npy',    label)


def process_split_local(chip_ids, s1_dir, s2_dir, label_dir, out_dir, split_name):
    os.makedirs(out_dir, exist_ok=True)
    errors  = []
    skipped = 0

    for chip_id in tqdm(chip_ids, desc=split_name):
        if os.path.exists(f'{out_dir}/{chip_id}_spatial.npy'):
            skipped += 1
            continue
        try:
            sp, tp, lb = process_chip_local(chip_id, s1_dir, s2_dir, label_dir)
            sp = normalize_spatial(sp).astype(np.float32)
            tp = normalize_temporal(tp).astype(np.float32)
            fill_nan_inf(sp)
            fill_nan_inf(tp)
            save_chip(chip_id, sp, tp, lb, out_dir)
        except Exception as e:
            errors.append((chip_id, str(e)))

    print(f'{split_name}: {len(chip_ids)-len(errors)-skipped} saved, '
          f'{skipped} skipped, {len(errors)} errors')
    if errors:
        for chip_id, err in errors[:10]:
            print(f'  {chip_id}: {err}')
    return errors


# Process train
train_errors = process_split_local(
    train_chips_l, TRAIN_S1_L, TRAIN_S2_L, TRAIN_LABEL_L,
    f'{OUT_BASE}/train', 'TRAIN'
)

# Process val
val_errors = process_split_local(
    val_chips_l, VAL_S1_L, VAL_S2_L, VAL_LABEL_L,
    f'{OUT_BASE}/val', 'VAL'
)

TRAIN: 100%|██████████| 2064/2064 [46:50<00:00,  1.36s/it]


TRAIN: 2064 saved, 0 skipped, 0 errors


VAL: 100%|██████████| 168/168 [03:40<00:00,  1.31s/it]

VAL: 168 saved, 0 skipped, 0 errors


***Verification***

Checks file counts, shapes, dtypes, NaN, and label values on a random sample.

In [ ]:
import random
print('=== Output Verification ===\n')

for split, expected_n in [('train', len(train_chips_l)), ('val', len(val_chips_l))]:
    out_dir  = f'{OUT_BASE}/{split}'
    sp_files = glob.glob(f'{out_dir}/*_spatial.npy')
    tp_files = glob.glob(f'{out_dir}/*_temporal.npy')
    lb_files = glob.glob(f'{out_dir}/*_label.npy')

    print(f'{split.upper()} (expected {expected_n} chips):')
    print(f'  spatial  : {len(sp_files)} files')
    print(f'  temporal : {len(tp_files)} files')
    print(f'  label    : {len(lb_files)} files')

    sample = random.sample(sp_files, min(3, len(sp_files)))
    for sp_path in sample:
        chip_id  = os.path.basename(sp_path).replace('_spatial.npy', '')
        sp  = np.load(f'{out_dir}/{chip_id}_spatial.npy')
        tp  = np.load(f'{out_dir}/{chip_id}_temporal.npy')
        lb  = np.load(f'{out_dir}/{chip_id}_label.npy')
        shape_ok = sp.shape==(14,512,512) and tp.shape==(15,2,512,512) and lb.shape==(512,512)
        dtype_ok = sp.dtype==np.float32 and tp.dtype==np.float32 and lb.dtype==np.int16
        nan_ok   = not np.isnan(sp).any() and not np.isnan(tp).any()
        label_ok = set(np.unique(lb)).issubset({-1,0,1})
        status   = '✅' if all([shape_ok,dtype_ok,nan_ok,label_ok]) else '❌'
        print(f'  {status} {chip_id}  shape={shape_ok}  dtype={dtype_ok}  no_nan={nan_ok}  labels={label_ok}')
    print()

# Storage used
total_bytes = sum(os.path.getsize(f) for f in glob.glob(f'{OUT_BASE}/**/*.npy', recursive=True))
print(f'Total preprocessed size: {total_bytes/1e9:.2f} GB')

=== Output Verification ===

TRAIN (expected 2064 chips):
  spatial  : 2064 files
  temporal : 2064 files
  label    : 2064 files
  ✅ Sri-Lanka_348501  shape=True  dtype=True  no_nan=True  labels=True
  ✅ India_3690927  shape=True  dtype=True  no_nan=True  labels=True
  ✅ Mekong_7163479  shape=True  dtype=True  no_nan=True  labels=True

VAL (expected 168 chips):
  spatial  : 168 files
  temporal : 168 files
  label    : 168 files
  ✅ Pakistan_548910  shape=True  dtype=True  no_nan=True  labels=True
  ✅ India_869358  shape=True  dtype=True  no_nan=True  labels=True
  ✅ Mekong_596495  shape=True  dtype=True  no_nan=True  labels=True

Total preprocessed size: 104.15 GB


In [ ]:
import os, glob

out_base = '/content/drive/MyDrive/FloodProject/data/preprocessed'
by_type = {'spatial': 0, 'temporal': 0, 'label': 0}

for f in glob.glob(f'{out_base}/**/*.npy', recursive=True):
    size = os.path.getsize(f)
    for t in by_type:
        if f.endswith(f'_{t}.npy'):
            by_type[t] += size

for t, size in by_type.items():
    print(f'{t:10s}: {size/1e9:.2f} GB')
print(f'Total     : {sum(by_type.values())/1e9:.2f} GB')

spatial   : 32.77 GB
temporal  : 70.21 GB
label     : 1.17 GB
Total     : 104.15 GB


In [ ]:
import os, glob

out_base = '/content/drive/MyDrive/FloodProject/data/preprocessed'
by_type = {'spatial': 0, 'temporal': 0, 'label': 0}

for f in glob.glob(f'{out_base}/**/*.npy', recursive=True):
    size = os.path.getsize(f)
    for t in by_type:
        if f.endswith(f'_{t}.npy'):
            by_type[t] += size

for t, size in by_type.items():
    print(f'{t:10s}: {size/1e9:.2f} GB')
print(f'Total     : {sum(by_type.values())/1e9:.2f} GB')

spatial   : 32.77 GB
temporal  : 70.21 GB
label     : 1.17 GB
Total     : 104.15 GB


In [ ]:
import os, glob

out_base = '/content/drive/MyDrive/FloodProject/data/preprocessed'

train_files = glob.glob(f'{out_base}/train/*.npy')
val_files   = glob.glob(f'{out_base}/val/*.npy')

types = {'spatial': 0, 'temporal': 0, 'label': 0}
for f in train_files + val_files:
    for t in types:
        if f.endswith(f'_{t}.npy'):
            types[t] += 1

print(f'Train files : {len(train_files)}')
print(f'Val files   : {len(val_files)}')
print(f'Total       : {len(train_files) + len(val_files)}')
print()
for t, n in types.items():
    print(f'  {t:10s}: {n}')

Train files : 6192
Val files   : 504
Total       : 6696

  spatial   : 2232
  temporal  : 2232
  label     : 2232


In [ ]:
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
import glob, os

# Check temporal values for a few chips
TEMPORAL_DIR = '/content/drive/MyDrive/FloodProject/data/sen1floods11/temporal'
VAL_S1       = '/content/drive/MyDrive/FloodProject/data/sen1floods11/val/S1Hand'

s1_files = glob.glob(f'{VAL_S1}/*.tif')[:5]

for s1_path in s1_files:
    chip_id = '_'.join(os.path.basename(s1_path).split('_')[:2])
    country = chip_id.split('_')[0]
    temporal_country = 'Cambodia' if country == 'Mekong' else country
    temp_path = f'{TEMPORAL_DIR}/{temporal_country}_TEMPORAL.tif'

    with rasterio.open(s1_path) as chip_src:
        dst_crs       = chip_src.crs
        dst_transform = chip_src.transform
        dst_height    = chip_src.height
        dst_width     = chip_src.width

    with rasterio.open(temp_path) as temp_src:
        src_data = temp_src.read().astype(np.float32)
        src_transform = temp_src.transform
        src_crs = temp_src.crs

    reprojected = np.zeros((30, dst_height, dst_width), dtype=np.float32)
    for b in range(30):
        reproject(
            source        = src_data[b],
            destination   = reprojected[b],
            src_transform = src_transform,
            src_crs       = src_crs,
            dst_transform = dst_transform,
            dst_crs       = dst_crs,
            resampling    = Resampling.bilinear,
            src_nodata    = temp_src.nodata,
            dst_nodata    = 0.0,
        )

    chirps = reprojected[0::2]  # all CHIRPS bands
    era5   = reprojected[1::2]  # all ERA5 bands

    print(f'{chip_id} ({temporal_country}):')
    print(f'  CHIRPS: min={chirps.min():.3f}  max={chirps.max():.3f}  mean={chirps.mean():.3f}  zeros={( chirps==0).mean()*100:.1f}%')
    print(f'  ERA5  : min={era5.min():.4f}  max={era5.max():.4f}  mean={era5.mean():.4f}  zeros={(era5==0).mean()*100:.1f}%')
    print()

India_1018317 (India):
  CHIRPS: min=0.000  max=23.571  mean=8.000  zeros=41.0%
  ERA5  : min=0.4444  max=0.4912  mean=0.4724  zeros=0.0%

India_1017769 (India):
  CHIRPS: min=0.000  max=26.443  mean=9.695  zeros=26.1%
  ERA5  : min=0.4470  max=0.4996  mean=0.4760  zeros=0.0%

India_1068117 (India):
  CHIRPS: min=0.000  max=36.506  mean=14.734  zeros=20.0%
  ERA5  : min=0.4538  max=0.4877  mean=0.4729  zeros=0.0%

India_118868 (India):
  CHIRPS: min=0.000  max=40.099  mean=15.296  zeros=20.0%
  ERA5  : min=0.4507  max=0.5013  mean=0.4760  zeros=0.0%

India_1072277 (India):
  CHIRPS: min=0.000  max=30.471  mean=9.607  zeros=20.0%
  ERA5  : min=0.4458  max=0.4947  mean=0.4703  zeros=0.0%



In [ ]:
import numpy as np, glob, os, random

out_base = '/content/drive/MyDrive/FloodProject/data/preprocessed'

# Sample 5 random temporal files from train and val
all_temporal = (glob.glob(f'{out_base}/train/*_temporal.npy') +
                glob.glob(f'{out_base}/val/*_temporal.npy'))

print(f'Total temporal files: {len(all_temporal)}')
print()

sample = random.sample(all_temporal, min(5, len(all_temporal)))

for fpath in sample:
    chip_id = os.path.basename(fpath).replace('_temporal.npy', '')
    arr     = np.load(fpath)  # expected (15, 2, 512, 512)

    chirps  = arr[:, 0, :, :]  # (15, 512, 512)
    era5    = arr[:, 1, :, :]  # (15, 512, 512)

    shape_ok = arr.shape == (15, 2, 512, 512)
    dtype_ok = arr.dtype == np.float32
    nan_ok   = not np.isnan(arr).any()

    # After normalization CHIRPS mean~0, ERA5 mean~0
    # Values should be in roughly [-3, 3] range for normalized data
    range_ok = arr.min() > -10 and arr.max() < 10

    status = '✅' if all([shape_ok, dtype_ok, nan_ok, range_ok]) else '❌'
    print(f'{status} {chip_id}')
    print(f'   shape={arr.shape}  dtype={arr.dtype}')
    print(f'   CHIRPS: min={chirps.min():.3f}  max={chirps.max():.3f}  mean={chirps.mean():.3f}  zeros={( chirps==0).mean()*100:.1f}%')
    print(f'   ERA5  : min={era5.min():.3f}  max={era5.max():.3f}  mean={era5.mean():.3f}  zeros={(era5==0).mean()*100:.1f}%')
    print(f'   NaN={np.isnan(arr).any()}  Inf={np.isinf(arr).any()}  range_ok={range_ok}')
    print()

Total temporal files: 2232

✅ Bolivia_6355611
   shape=(15, 2, 512, 512)  dtype=float32
   CHIRPS: min=-0.765  max=5.946  mean=0.122  zeros=0.0%
   ERA5  : min=0.212  max=0.709  mean=0.554  zeros=0.0%
   NaN=False  Inf=False  range_ok=True

✅ India_1692173
   shape=(15, 2, 512, 512)  dtype=float32
   CHIRPS: min=-0.765  max=2.962  mean=0.667  zeros=0.0%
   ERA5  : min=0.388  max=0.790  mean=0.611  zeros=0.0%
   NaN=False  Inf=False  range_ok=True

✅ India_5314754
   shape=(15, 2, 512, 512)  dtype=float32
   CHIRPS: min=-0.765  max=2.620  mean=0.312  zeros=0.0%
   ERA5  : min=-0.287  max=0.443  mean=0.109  zeros=0.0%
   NaN=False  Inf=False  range_ok=True

✅ Pakistan_163672
   shape=(15, 2, 512, 512)  dtype=float32
   CHIRPS: min=-0.765  max=0.169  mean=-0.605  zeros=0.0%
   ERA5  : min=-2.666  max=-0.976  mean=-2.156  zeros=0.0%
   NaN=False  Inf=False  range_ok=True

✅ Bolivia_4138146
   shape=(15, 2, 512, 512)  dtype=float32
   CHIRPS: min=-0.765  max=5.177  mean=0.510  zeros=0.0%
  

In [ ]:
import numpy as np, glob, os
from collections import Counter

out_base = '/content/drive/MyDrive/FloodProject/data/preprocessed'

all_temporal = (glob.glob(f'{out_base}/train/*_temporal.npy') +
                glob.glob(f'{out_base}/val/*_temporal.npy'))

# Count by country
countries = Counter()
for f in all_temporal:
    chip_id = os.path.basename(f).replace('_temporal.npy', '')
    country = chip_id.split('_')[0]
    countries[country] += 1

print('Temporal files by country:')
for country, n in sorted(countries.items()):
    print(f'  {country:12s}: {n}')
print(f'  {"TOTAL":12s}: {sum(countries.values())}')

# Sample one chip per country and check values
print('\nOne sample per country:')
seen = set()
for f in sorted(all_temporal):
    chip_id = os.path.basename(f).replace('_temporal.npy', '')
    country = chip_id.split('_')[0]
    if country in seen:
        continue
    seen.add(country)
    arr    = np.load(f)
    chirps = arr[:, 0, :, :]
    era5   = arr[:, 1, :, :]
    print(f'  {country:12s}: CHIRPS mean={chirps.mean():>7.3f}  ERA5 mean={era5.mean():>7.3f}  NaN={np.isnan(arr).any()}')

Temporal files by country:
  Bolivia     : 224
  Colombia    : 534
  India       : 535
  Mekong      : 430
  Pakistan    : 277
  Sri-Lanka   : 232
  TOTAL       : 2232

One sample per country:
  Bolivia     : CHIRPS mean=  0.142  ERA5 mean=  0.540  NaN=False
  Colombia    : CHIRPS mean=  0.368  ERA5 mean=  0.577  NaN=False
  India       : CHIRPS mean=  0.441  ERA5 mean=  0.616  NaN=False
  Mekong      : CHIRPS mean=  0.006  ERA5 mean=  0.577  NaN=False
  Pakistan    : CHIRPS mean= -0.608  ERA5 mean= -1.726  NaN=False
  Sri-Lanka   : CHIRPS mean=  1.604  ERA5 mean=  0.682  NaN=False


In [ ]:
import os, glob, shutil
from tqdm import tqdm

out_base = '/content/drive/MyDrive/FloodProject/data/preprocessed'

os.makedirs(f'{out_base}/train_temporal', exist_ok=True)
os.makedirs(f'{out_base}/val_temporal',   exist_ok=True)

for split, temp_dir in [('train', 'train_temporal'), ('val', 'val_temporal')]:
    files = glob.glob(f'{out_base}/{split}/*_temporal.npy')
    print(f'Moving {len(files)} files from {split}/ → {temp_dir}/')
    for f in tqdm(files, desc=split):
        shutil.move(f, f'{out_base}/{temp_dir}/{os.path.basename(f)}')

print('Done.')

Moving 2064 files from train/ → train_temporal/


train: 100%|██████████| 2064/2064 [00:10<00:00, 200.14it/s]


Moving 168 files from val/ → val_temporal/


val: 100%|██████████| 168/168 [00:00<00:00, 237.19it/s]

Done.
